# Intermediate 04 — Central Limit Theorem and Delta Method

Practice notebook for the **"Central Limit Theorem and Delta Method"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


## Part 1 — The classical Central Limit Theorem

The PDF states the **Lindeberg-Levy CLT**: if $X_1,\dots,X_n$ are i.i.d. with
$E[X_i]=\mu$ and $\operatorname{Var}(X_i)=\sigma^2<\infty$, then with
$Z_n=(\bar X_n-\mu)/(\sigma/\sqrt{n})$,

$$
Z_n \;\Rightarrow\; \mathcal{N}(0,1).
$$

In practice, for large $n$ the sample mean satisfies

$$
\bar X_n \;\approx\; \mathcal{N}\!\left(\mu,\, \frac{\sigma^2}{n}\right).
$$

This is why the standard error of a sample mean is $\sigma/\sqrt{n}$ and why a
normal approximation is used for many test statistics.

Below we draw samples from a clearly **non-normal** distribution —
$X_i\sim\text{Exponential}(\lambda=1)$, which has $\mu=1$ and $\sigma^2=1$ — and
check that the histogram of $\bar X_n$ across many replications lines up with the
$\mathcal{N}(1, 1/n)$ density predicted by the CLT.


In [ ]:
# Exponential(lambda=1): mu = 1, sigma^2 = 1 (non-normal!)
mu = 1.0
sigma2 = 1.0

rng = np.random.default_rng(7)
n = 30              # sample size per replication
R = 50_000          # number of replications

# R sample means, each from n iid Exponential(1) draws
samples = rng.exponential(scale=1.0, size=(R, n))
xbar = samples.mean(axis=1)

# CLT prediction: N(mu, sigma^2 / n)
clt_mean = mu
clt_std = np.sqrt(sigma2 / n)

xs = np.linspace(xbar.min(), xbar.max(), 400)
plt.hist(xbar, bins=80, density=True, alpha=0.55, label=f"histogram of $\\bar X_n$, n={n}")
plt.plot(xs, stats.norm.pdf(xs, loc=clt_mean, scale=clt_std),
         "r-", lw=2, label=f"CLT $\\mathcal{{N}}({mu}, {sigma2}/{n})$")
plt.axvline(mu, color="k", linestyle="--", alpha=0.6, label=f"true $\\mu={mu}$")
plt.xlabel("$\\bar X_n$")
plt.ylabel("density")
plt.title("CLT: sample means of Exponential(1) are approximately normal")
plt.legend()
plt.tight_layout()
plt.show()

print(f"sample mean of xbar   = {xbar.mean():.4f}  (theory {mu})")
print(f"sample var  of xbar   = {xbar.var(ddof=1):.5f} (theory {sigma2/n:.5f})")


## Part 2 — Convergence improves with $n$

The CLT is an *asymptotic* statement: the normal approximation gets better as $n$
grows. With small $n$ the histogram of $\bar X_n$ inherits skew from the parent
distribution; as $n\to\infty$ it tightens and symmetrizes around $\mu$, with
variance shrinking like $1/n$.

We visualize this by repeating the experiment of Part 1 for several values of $n$
and overlaying each CLT density $\mathcal{N}(\mu, \sigma^2/n)$.


In [ ]:
rng = np.random.default_rng(11)
R = 50_000
ns = [1, 5, 30, 120]
colors = ["tab:red", "tab:orange", "tab:green", "tab:blue"]

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharey=False)
for ax, n, c in zip(axes.ravel(), ns, colors):
    samples = rng.exponential(scale=1.0, size=(R, n))
    xbar = samples.mean(axis=1)
    clt_std = np.sqrt(sigma2 / n)
    xs = np.linspace(xbar.min(), xbar.max(), 400)
    ax.hist(xbar, bins=60, density=True, alpha=0.5, color=c,
            label=f"histogram, n={n}")
    ax.plot(xs, stats.norm.pdf(xs, loc=mu, scale=clt_std),
            "k-", lw=2, label=f"CLT $\\mathcal{{N}}({mu}, {sigma2}/{n})$")
    ax.axvline(mu, color="gray", linestyle="--", alpha=0.7)
    ax.set_title(f"n = {n}")
    ax.set_xlabel("$\\bar X_n$")
    ax.legend(fontsize=8)
fig.suptitle("Convergence of $\\bar X_n$ to a normal as n grows (Exponential parent)")
plt.tight_layout()
plt.show()

# Quantify: standardized 3rd moment (skew) of xbar should -> 0 as n grows
print(f"{'n':>5} {'std(xbar)':>12} {'theory':>10} {'skew(xbar)':>12}")
for n in ns:
    s = rng.exponential(scale=1.0, size=(R, n)).mean(axis=1)
    print(f"{n:>5} {s.std(ddof=1):>12.5f} {np.sqrt(sigma2/n):>10.5f} "
          f"{stats.skew(s):>12.5f}")


## Part 3 — The Delta Method (first-order)

The CLT tells us about $\bar X_n$ itself. The **delta method** propagates that
asymptotic normality through a smooth function $g$. From the PDF: if
$\sqrt{n} (T_n - \theta) \Rightarrow \mathcal{N}(0, \tau^2)$ and $g$ is differentiable
at $\theta$ with $g'(\theta)\ne 0$, then

$$
\sqrt{n}\big(g(T_n)-g(\theta)\big) \;\Rightarrow\;
\mathcal{N}\!\left(0,\, (g'(\theta))^2\,\tau^2\right).
$$

The heuristic is a first-order Taylor expansion:
$g(T_n)\approx g(\theta)+g'(\theta)(T_n-\theta)$, so

$$
\operatorname{Var}\big(g(T_n)\big) \;\approx\; \big(g'(\theta)\big)^2\,\frac{\sigma^2}{n}.
$$

We apply this to $g(x)=1/x$ on the sample mean of iid
$X_i\sim\text{Exponential}(1)$, where $\theta=\mu=1$, $\sigma^2=1$,
$g'(x)=-1/x^2$, so $g'(\theta)=-1$ and the delta-method variance is
$(-1)^2\cdot 1/n = 1/n$.


In [ ]:
# Delta method for g(x) = 1/x on xbar from Exponential(1)
# theta = mu = 1, sigma^2 = 1, g'(theta) = -1/theta^2 = -1
mu = 1.0
sigma2 = 1.0
g = lambda x: 1.0 / x
gp = lambda x: -1.0 / x**2

rng = np.random.default_rng(23)
R = 100_000
ns = [10, 50, 200, 1000]

print(f"{'n':>6} {'Var(g(Xbar)) sim':>18} {'delta approx (1/n)':>20} {'rel error':>12}")
for n in ns:
    xbar = rng.exponential(scale=1.0, size=(R, n)).mean(axis=1)
    gx = g(xbar)
    var_sim = gx.var(ddof=1)
    delta_var = (gp(mu) ** 2) * sigma2 / n   # = 1/n
    rel = abs(var_sim - delta_var) / delta_var
    print(f"{n:>6} {var_sim:>18.6f} {delta_var:>20.6f} {rel:>12.4%}")


## Part 4 — Variance-stabilizing transform for a Poisson mean

The PDF's worked example takes $X_i\sim\text{Poisson}(\lambda)$ so that
$\bar X_n$ has $\tau^2=\lambda$, and chooses $g(\theta)=2\sqrt\theta$ with
$g'(\theta)=1/\sqrt\theta$. Then

$$
\sqrt{n}\big(2\sqrt{\bar X_n}-2\sqrt\lambda\big)
\;\Rightarrow\; \mathcal{N}\!\left(0,\,\frac{1}{\lambda}\cdot\lambda\right)
= \mathcal{N}(0,1),
$$

i.e. $2\sqrt{\bar X_n}$ is approximately normal with mean $2\sqrt\lambda$ and a
**constant** variance $1/n$ independent of $\lambda$. This is the classical
variance-stabilizing transform for Poisson-like count data.

We simulate to confirm both the mean and the stabilized variance, comparing
$\operatorname{Var}(2\sqrt{\bar X_n})$ against $1/n$ and against the much
larger, $\lambda$-dependent $\operatorname{Var}(\bar X_n)=\lambda/n$.


In [ ]:
lam = 5.0
g_pois = lambda x: 2.0 * np.sqrt(x)

rng = np.random.default_rng(29)
R = 100_000
ns = [10, 50, 200, 1000]

print("Poisson lambda =", lam)
print(f"{'n':>6} {'Var(2sqrt(Xbar))':>18} {'1/n (theory)':>14} {'Var(Xbar)=lam/n':>17}")
for n in ns:
    xbar = rng.poisson(lam=lam, size=(R, n)).mean(axis=1)
    var_stab = g_pois(xbar).var(ddof=1)
    print(f"{n:>6} {var_stab:>18.6f} {1.0/n:>14.6f} {lam/n:>17.6f}")

# Visualize: histogram of the stabilized statistic vs N(2*sqrt(lam), 1/n)
n = 200
xbar = rng.poisson(lam=lam, size=(R, n)).mean(axis=1)
stat = g_pois(xbar)
xs = np.linspace(stat.min(), stat.max(), 400)
plt.hist(stat, bins=80, density=True, alpha=0.55,
         label=f"histogram of $2\\sqrt{{\\bar X_n}}$, n={n}")
plt.plot(xs, stats.norm.pdf(xs, loc=2*np.sqrt(lam), scale=np.sqrt(1.0/n)),
         "r-", lw=2, label=f"$\\mathcal{{N}}(2\\sqrt\\lambda, 1/n)$")
plt.axvline(2*np.sqrt(lam), color="k", linestyle="--", alpha=0.6,
            label=f"mean $2\\sqrt\\lambda$={2*np.sqrt(lam):.3f}")
plt.xlabel("$2\\sqrt{\\bar X_n}$")
plt.title("Variance-stabilizing transform for Poisson mean")
plt.legend()
plt.tight_layout()
plt.show()


## Part 5 — Delta method for $g(x)=\sqrt{x}$: simulated vs approximation

We close by applying the delta method to $g(x)=\sqrt{x}$ on the sample mean of
$X_i\sim\text{Exponential}(1)$. Here $\theta=\mu=1$, $\sigma^2=1$,
$g'(\theta)=\tfrac{1}{2\sqrt\theta}=\tfrac12$, so the delta-method variance is
$(\tfrac12)^2\cdot 1/n = \tfrac{1}{4n}$.

We compare the simulated $\operatorname{Var}(g(\bar X_n))$ against the
delta-method approximation $\tfrac{1}{4n}$ across $n$, and plot the
$\mathcal{N}(\sqrt\mu, 1/(4n))$ density over the histogram for one choice of $n$.

**Your turn:** change the parent distribution (e.g. `rng.gamma`) and the function
$g$ (try $g(x)=\log x$), then re-derive $g'(\theta)$ and check whether the
delta-method variance still tracks the simulated variance. Where does the
approximation break down, and why?


In [ ]:
mu = 1.0
sigma2 = 1.0
g = lambda x: np.sqrt(x)
gp = lambda x: 0.5 / np.sqrt(x)

rng = np.random.default_rng(53)
R = 200_000
ns = [5, 20, 80, 320, 1280]

print(f"{'n':>6} {'Var(sqrt(Xbar)) sim':>20} {'delta approx 1/(4n)':>20} {'rel error':>12}")
for n in ns:
    xbar = rng.exponential(scale=1.0, size=(R, n)).mean(axis=1)
    var_sim = g(xbar).var(ddof=1)
    delta_var = (gp(mu) ** 2) * sigma2 / n   # = 1/(4n)
    rel = abs(var_sim - delta_var) / delta_var
    print(f"{n:>6} {var_sim:>20.7f} {delta_var:>20.7f} {rel:>12.4%}")

# Overlay the delta-method density for one n
n = 80
xbar = rng.exponential(scale=1.0, size=(R, n)).mean(axis=1)
gs = g(xbar)
xs = np.linspace(gs.min(), gs.max(), 400)
plt.hist(gs, bins=80, density=True, alpha=0.55,
         label=f"histogram of $\\sqrt{{\\bar X_n}}$, n={n}")
plt.plot(xs, stats.norm.pdf(xs, loc=g(mu), scale=np.sqrt((gp(mu)**2)*sigma2/n)),
         "r-", lw=2,
         label=f"delta $\\mathcal{{N}}(\\sqrt\\mu, 1/(4n))$")
plt.axvline(g(mu), color="k", linestyle="--", alpha=0.6, label=f"$g(\\mu)=1$")
plt.xlabel("$\\sqrt{{\\bar X_n}}$")
plt.title("Delta method for $g(x)=\\sqrt{x}$ on Exponential(1) sample means")
plt.legend()
plt.tight_layout()
plt.show()


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. Let $X_1,\dots,X_n$ be i.i.d. $\text{Exponential}(\lambda=2)$ (so $\mu=1/2$, $\sigma^2=1/4$). State the CLT-based approximate distribution of $\bar X_n$ for large $n$ and give an approximate 95% confidence interval for $\mu$.

2. For $X_i\sim\text{Exponential}(1)$ with $g(x)=1/x$, derive the delta-method variance of $g(\bar X_n)$ and confirm numerically (modifying Part 3) that the simulated variance is close to $1/n$ for $n=200$.

3. Let $X_i\sim\text{Poisson}(\lambda)$. Derive the delta-method distribution of $\log \bar X_n$ for large $n$. What is its asymptotic variance, and why is $2\sqrt{x}$ preferred as a variance-stabilizing transform?

4. Suppose $\sqrt{n} (T_n-\theta)\Rightarrow \mathcal N(0,\tau^2)$ and you want a confidence interval for $g(\theta)=\theta^2$ (with $\theta\ne 0$). Use the delta method to give the asymptotic distribution of $g(T_n)$ and the resulting approximate $1-\alpha$ CI.

5. The delta method uses a *first-order* Taylor expansion. Sketch what changes at second order (the form of the bias correction $\tfrac12 g''(\theta)\operatorname{Var}(T_n)$) and briefly explain when a second-order correction matters most.


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** The CLT gives $\bar X_n\approx \mathcal N(\mu, \sigma^2/n)$, so here
$\bar X_n\approx \mathcal N(1/2,\, 1/(4n))$. An approximate 95% CI is
$\bar X_n \pm 1.96\,\frac{\sigma}{\sqrt{n}}=\bar X_n\pm 1.96\cdot\frac{1/2}{\sqrt{n}}$.
In practice $\sigma$ is replaced by the sample standard deviation, yielding the usual
$t$-interval.

**2.** With $\theta=\mu=1$, $\sigma^2=1$, $g(x)=1/x$, $g'(x)=-1/x^2$, so
$g'(\theta)=-1$ and the delta-method variance is $(-1)^2\cdot 1/n=1/n$.
Running the Part 3 code with $n=200$ gives a simulated variance of roughly
$0.005$ vs the theoretical $1/200=0.005$ (numbers vary slightly with seed), confirming
the approximation.

**3.** For $\bar X_n$ from Poisson$(\lambda)$, $\tau^2=\lambda$. With $g(x)=\log x$,
$g'(\lambda)=1/\lambda$, so
$\sqrt{n}(\log \bar X_n-\log\lambda)\Rightarrow \mathcal N(0,\,(1/\lambda)^2\lambda)=\mathcal N(0,1/\lambda)$.
The asymptotic variance $1/\lambda$ still depends on $\lambda$, so the log does *not*
stabilize variance. With $g(x)=2\sqrt{x}$, $g'(\lambda)=1/\sqrt\lambda$ and the
$\lambda$ cancels, giving asymptotic variance $1$; that is why $2\sqrt{x}$ is the
variance-stabilizing transform for Poisson data.

**4.** $g'(\theta)=2\theta$, so
$\sqrt{n} (T_n^2-\theta^2)\Rightarrow \mathcal N(0,\,4\theta^2\tau^2)$ and
$T_n^2\approx \mathcal N(\theta^2,\,4\theta^2\tau^2/n)$. An approximate
$1-\alpha$ CI is $T_n^2 \pm z_{1-\alpha/2}\cdot 2|\theta|\,\tau/\sqrt{n}$ (with
$\theta,\tau$ replaced by consistent estimates). Note the approximation fails near
$\theta=0$, where $g'(\theta)=0$ and the first-order delta method breaks down.

**5.** A second-order expansion gives
$g(T_n)\approx g(\theta)+g'(\theta)(T_n-\theta)+\tfrac12 g''(\theta)(T_n-\theta)^2$.
Taking expectations, $E[g(T_n)]\approx g(\theta)+\tfrac12 g''(\theta)\operatorname{Var}(T_n)$,
so the bias is approximately $\tfrac12 g''(\theta)\,\sigma^2/n$. A second-order
correction matters most when $g'\approx 0$ (the linear term vanishes and the
quadratic term dominates) or when $n$ is only moderately large, so the linear
approximation alone is not yet accurate.

</details>
